# 🇹🇷 Emlak Asistanı — LoRA Fine-Tuning (Unsloth + LLaMA-3)

Bu notebook Türk emlak ilanlarından oluşturulmuş bir veri setiyle LLaMA-3 modelini LoRA ile ince ayar yapar. Amaç modelin tamamen Türkçe yanıt vermesini sağlamak (İngilizce sözcük kalmaması).

### Adımlar
1. Bağımlılıkları kur
2. Eğitim verisini Google Drive'dan yükle
3. Modeli ve LoRA adaptörlerini yükle
4. Eğitimi başlat (SFTTrainer)
5. Modeli kaydet ve test et

### Gereksinimler
- **Runtime**: GPU (Colab → Çalışma zamanı → GPU T4 veya A100)
- **HuggingFace Token**: `hf_...` ile başlayan token gerekebilir (LLaMA modelini indirmek için)


In [ ]:
# ============================================================
# HÜCRE 1: Bağımlılıkları Kur
# ============================================================
!pip install unsloth trl datasets transformers peft accelerate -q
print('✅ Kurulum tamamlandı.')

In [ ]:
# ============================================================
# HÜCRE 2: Google Drive Bağla & Veriyi Yükle
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import json

# ⚠️ finetune_dataset.jsonl dosyanızın Drive'daki tam yolunu buraya yazın:
DATASET_PATH = '/content/drive/MyDrive/emlak_lora/finetune_dataset.jsonl'

print("Dosya var mı?", os.path.exists(DATASET_PATH))
print("Dosya yolu:", DATASET_PATH)

records = []
if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    print(f'✅ {len(records)} eğitim örneği yüklendi.')
    print('--- İlk örnek ---')
    for m in records[0]['messages']:
        print(f"[{m['role'].upper()}]\n{m['content'][:500]}\n")
else:
    print("❌ HATA: Dosya bulunamadı! Lütfen yolu kontrol edin.")

In [ ]:
# ============================================================
# HÜCRE 3: Modeli Yükle (Unsloth + LoRA)
# ============================================================
from unsloth import FastLanguageModel
import torch

# Model seçenekleri (LoRA ile çalışan küçük modeller):
# - "unsloth/llama-3-8b-instruct-bnb-4bit"   (LLaMA-3 8B, önerilen)
MODEL_NAME = 'unsloth/llama-3-8b-instruct-bnb-4bit'

MAX_SEQ_LEN = 512    # Bellek tasarrufu için 1024'ten 512'ye düşürüldü
DTYPE = None         # Otomatik
LOAD_IN_4BIT = True  # Bellek tasarrufu için 4-bit niceleme

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

# LoRA Adaptör Konfigürasyonu
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
print('✅ Model ve LoRA adaptörleri yüklendi.')
model.print_trainable_parameters()

In [ ]:
# ============================================================
# HÜCRE 4: Veri Setini Hazırla (ChatML formatı)
# ============================================================
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template='llama-3',
    mapping={'role': 'role', 'content': 'content', 'user': 'user', 'assistant': 'assistant'},
)

def format_prompt(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

raw_dataset = Dataset.from_list(records)
dataset = raw_dataset.map(format_prompt, batched=False)

split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
eval_ds  = split['test']

print(f'✅ Eğitim: {len(train_ds)} | Doğrulama: {len(eval_ds)}')
print('\nÖrnek metin (ilk 500 karakter):')
print(train_ds[0]['text'][:500])

In [ ]:
# ============================================================
# HÜCRE 5: Eğitim Konfigürasyonu ve SFTTrainer
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

OUTPUT_DIR = '/content/emlak_lora_model'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,  # T4 bellek hatası için 4'ten 2'ye düşürüldü
    gradient_accumulation_steps=8, # Toplam batch size (2x8=16) korunmak için 4'ten 8'e çıkarıldı
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print('✅ Trainer hazır. Eğitim başlıyor...')
trainer_stats = trainer.train()
print(f'✅ Eğitim tamamlandı! Loss: {trainer_stats.training_loss:.4f}')

In [ ]:
# ============================================================
# HÜCRE 6: Modeli Kaydet
# ============================================================
model.save_pretrained(OUTPUT_DIR + '/lora_adapter')
tokenizer.save_pretrained(OUTPUT_DIR + '/lora_adapter')
print(f'✅ LoRA adaptörü kaydedildi: {OUTPUT_DIR}/lora_adapter')

import shutil
DRIVE_SAVE_PATH = '/content/drive/MyDrive/emlak_lora/lora_adapter'
shutil.copytree(OUTPUT_DIR + '/lora_adapter', DRIVE_SAVE_PATH, dirs_exist_ok=True)
print(f'✅ Adaptör Drive\'a kaydedildi: {DRIVE_SAVE_PATH}')

In [ ]:
# ============================================================
# HÜCRE 7: Model Testi
# ============================================================
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

TEST_MESSAGES = [
    {
        'role': 'system',
        'content': (
            'Sen bir Türk emlak danışmanı asistanısın. '
            'Yalnızca Türkçe yaz. Asla İngilizce kelime kullanma.'
        )
    },
    {
        'role': 'user',
        'content': 'Kadıköy\'de 2.5m TL altı 3+1 uygun mu?'
    }
]

inputs = tokenizer.apply_chat_template(
    TEST_MESSAGES,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to('cuda')

outputs = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
response_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print('=== Model Yanıtı ===')
print(response_text)